<a href="https://colab.research.google.com/github/charlie-aashutosh/100-Next-Gen-Engg-C5/blob/main/RAG_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-google-genai langchain-community langchain-core pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstal

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

In [3]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

Enter your API Key··········


In [4]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

In [5]:
from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file

/tmp/ipykernel_688/4271358169.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file


In [6]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")

In [7]:
documents=loader.load()

report=""

for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))

Total number of pages 9


In [8]:
print(report[:1000]) #first 1000 tokens from the document

AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI trainin

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #helps in chunking

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, #inside each chunk, how many tokens we have ?
    chunk_overlap=200 #how many tokens are overlapping between each chunk ?
)

chunks=text_splitter.split_documents(documents)

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [11]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [12]:
query="What all investments did AxiOraa make in AI?"

In [16]:
results=vector_store.similarity_search_with_score(query=query)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score:{score}")
  print(f"Document:{doc.page_content[:500]}")

RESULT0
Score:0.5520483255386353
Document:AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrati
RESULT1
Score:0.5741000175476074
Document:engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insur

In [17]:
retriever=vector_store.as_retriever(
    search_type="similarity", #performing similarity search b/w vector embeddings and query embeddings
    search_kwargs={"k":1} #generating top 1 response
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate

In [19]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Management Consultant and your job is to answer
    the user questions based on the retrieved context.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:{context}
    Question:{question}

    Provide:
    1. Answer
    2. Supporting Evidence
    """
)

In [21]:
from langchain_core.output_parsers import StrOutputParser

In [22]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [23]:
question="Talk about the business overview of this company"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [24]:
print(response)

**1. Answer**
Based on the provided context, the company operates across four primary service lines: Strategy Consulting, AI Engineering, Managed Analytics, and Intelligent Automation. Its key geographic markets include the Asia-Pacific (APAC) region, Europe, and North America. The business profile includes narrative analysis, management commentary, illustrative metrics, assumptions, trends, benchmark observations, and fictional disclosures; however, all figures provided are synthetic and created strictly for educational and AI training purposes.

**2. Supporting Evidence**
* "The company operates across Strategy Consulting, AI Engineering, Managed Analytics and Intelligent Automation."
* "Major markets include APAC, Europe and North America."
* "This section includes narrative analysis, management commentary, illustrative metrics, assumptions, trends, benchmark observations, and fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended solely fo

In [25]:
question="help me understand the financial and operational highlights of this company"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [26]:
print(response)

**1. Answer**

Based on the provided financial and operational context, the company demonstrated strong financial growth and profitability expansion:

* **Top-line Growth:** Revenue increased by 21.1%, growing from USD 2.18 billion to USD 2.64 billion.
* **Profitability Expansion:** 
  * Gross Margin improved by 2.7 percentage points, moving from 41.5% to 44.2%.
  * EBITDA Margin reached a strong 22.8%.
  * Net Profit increased to USD 352 million.
* **Cash Flow & Liquidity:** The company maintained a healthy liquidity position with strong Free Cash Flow.

---

**2. Supporting Evidence**

The following direct excerpts from the provided context support these insights:

* *"Revenue grew from USD 2.18B to USD 2.64B (21.1%)."*
* *"Gross Margin improved from 41.5% to 44.2%."*
* *"EBITDA Margin reached 22.8%."*
* *"Net Profit increased to USD 352M."*
* *"Free Cash Flow remained strong."*
